# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [ ]:
import sys
import os

# Ensure src/ is on the path so `lasr` is importable.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from lasr.config import ModelConfig, InferenceConfig, PromptStyle, SAEConfig
from lasr.data import load_esnli, build_few_shot_examples, build_prompts
from lasr.inference import load_model, generate_predictions
from lasr.metrics import run_evaluation
from lasr.sae import load_sae
from lasr.activations import gather_residual_activations, encode_activations
from lasr.aggregation import top_k_features, top_k_features_per_token, reconstruction_metrics, l0_sparsity
from lasr.plotting import plot_feature_activation_heatmap, plot_per_token_topk_heatmap

# Configuration

In [ ]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT
use_few_shot = True

sae_config = SAEConfig(
    layer=40,
    width="65k",
    l0="medium",
    repo_id="google/gemma-scope-2-27b-it",
)

ESNLI_URL = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/refs/heads/master/dataset/esnli_dev.csv"

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

# Setup — HF Token

In [ ]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [ ]:
esnli_df = load_esnli(ESNLI_URL)
esnli_df.head()

# Build Prompts

In [ ]:
few_shot_examples = build_few_shot_examples(esnli_df, prompt_style) if use_few_shot else None
esnli_df["prompt"] = build_prompts(esnli_df, prompt_style, few_shot=use_few_shot, few_shot_examples=few_shot_examples)

print(esnli_df["prompt"].iloc[0])

# Load Model

In [ ]:
model, tokenizer = load_model(model_config)

# Inference — Chain of Thought (Few-shot)

In [ ]:
valid_mask = esnli_df["prompt"].notna()
valid_indices = esnli_df.index[valid_mask][:: inference_config.downsample_rate]
sampled_mask = esnli_df.index.isin(valid_indices)

prompts = esnli_df.loc[sampled_mask, "prompt"].tolist()
print(f"Running on {len(prompts)} / {valid_mask.sum()} samples (1 in every {inference_config.downsample_rate})")

decoded_outputs = generate_predictions(
    prompts, model, tokenizer, inference_config, device=model_config.device
)

esnli_df, report = run_evaluation(
    esnli_df, decoded_outputs, sampled_mask, prompt_style,
    model_name=model_config.model_name, few_shot=use_few_shot,
)
print(report)

# Inference — Chain of Thought (Zero-shot)

In [ ]:
cot_zs_prompt_style = PromptStyle.CHAIN_OF_THOUGHT
cot_zs_few_shot = False

esnli_df["prompt"] = build_prompts(esnli_df, cot_zs_prompt_style, few_shot=cot_zs_few_shot)

cot_zs_prompts = esnli_df.loc[sampled_mask, "prompt"].tolist()
print(f"Running on {len(cot_zs_prompts)} / {valid_mask.sum()} samples (1 in every {inference_config.downsample_rate})")

cot_zs_decoded_outputs = generate_predictions(
    cot_zs_prompts, model, tokenizer, inference_config, device=model_config.device
)

esnli_df, cot_zs_report = run_evaluation(
    esnli_df, cot_zs_decoded_outputs, sampled_mask, cot_zs_prompt_style,
    model_name=model_config.model_name, few_shot=cot_zs_few_shot,
)
print(cot_zs_report)

# SAE Activation Analysis

In [ ]:
sae = load_sae(sae_config)

## Gather activations for generated tokens

In [ ]:
import torch

# Pick the first sample prompt
sample_prompt = esnli_df["prompt"].iloc[0]
prompt_ids = tokenizer.encode(sample_prompt, return_tensors="pt", add_special_tokens=True).to(model_config.device)
prompt_len = prompt_ids.shape[1]
print(f"Prompt tokens: {prompt_len}")

# Generate the model's response (returns prompt + generated tokens)
full_ids = model.generate(input_ids=prompt_ids, max_new_tokens=inference_config.max_new_tokens)
gen_len = full_ids.shape[1] - prompt_len
print(f"Generated tokens: {gen_len}")
print(f"Full sequence tokens: {full_ids.shape[1]}")

# Run a forward pass on the full sequence and hook the SAE target layer
residual_acts = gather_residual_activations(model, sae_config.layer, full_ids)
print(f"Residual activations shape (full): {residual_acts.shape}")

# Slice to keep only the generated-token activations
gen_acts = residual_acts[:, prompt_len:, :]
print(f"Generated-only activations shape: {gen_acts.shape}")

# Encode through the SAE
sae_acts, reconstruction = encode_activations(sae, gen_acts)
print(f"SAE feature activations shape: {sae_acts.shape}")
print(f"Reconstruction shape: {reconstruction.shape}")

In [ ]:
# Reconstruction quality
metrics = reconstruction_metrics(reconstruction, gen_acts)
print(f"MSE: {metrics['mse']:.6f}")
print(f"FVU: {metrics['fvu']:.6f}")

In [ ]:
# L0 sparsity
l0 = l0_sparsity(sae_acts)
print(f"Per-token L0: {l0}")
print(f"Average L0: {l0.float().mean():.1f}")

In [ ]:
# Top-K features per token
from lasr.neuronpedia import get_neuronpedia_labels, build_sae_id

TOP_K = 10
per_token_vals, per_token_idxs = top_k_features_per_token(sae_acts, k=TOP_K)
print(f"Per-token top-{TOP_K} values shape: {per_token_vals.shape}")
print(f"Per-token top-{TOP_K} indices shape: {per_token_idxs.shape}")

# Collect all unique feature indices across every token
unique_features = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
print(f"Unique features across all tokens: {len(unique_features)}")

# Fetch concept labels from Neuronpedia for every unique feature
np_model_id = "gemma-3-27b-it"
np_sae_id = build_sae_id(sae_config)
labels = get_neuronpedia_labels(np_model_id, np_sae_id, unique_features)

print(f"\nTop {TOP_K} SAE features per token (first 3 tokens shown):")
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)
for t in range(min(3, len(tokens))):
    print(f"\n  Token '{tokens[t]}':")
    for r in range(TOP_K):
        idx = int(per_token_idxs[t, r])
        val = float(per_token_vals[t, r])
        concept = labels.get(idx)
        concept_str = f"  | concept: {concept}" if concept else ""
        print(f"    Rank {r+1}  feature {idx:>5d}  activation = {val:.4f}{concept_str}")

## Activation Heatmap

In [ ]:
fig = plot_per_token_topk_heatmap(
    per_token_vals,
    per_token_idxs,
    tokens=tokens,
    labels=labels,
    title="Gemma3-27b-it Per-Token Top-K SAE Feature Activations (Layer 40)",
)
fig.show()

## Inspect Feature — Neuronpedia Embed + Token Highlighting

In [ ]:
from IPython.display import display, HTML, IFrame
from lasr.neuronpedia import get_neuronpedia_feature_urls
import numpy as np


def inspect_feature(feature_idx: int):
    """Display the Neuronpedia feature page in an IFrame and show tokens
    highlighted by activation strength for that feature.

    Uses ``sae_acts``, ``tokens``, ``np_model_id``, ``np_sae_id``, and
    ``labels`` from the notebook scope.
    """
    # --- IFrame embed ---
    urls = get_neuronpedia_feature_urls(
        np_model_id, np_sae_id, [feature_idx], embed=True
    )
    embed_url = urls[feature_idx]
    display(IFrame(src=embed_url, width=620, height=480))

    # --- Token highlighting ---
    # sae_acts shape: (1, n_tokens, n_features)
    acts = sae_acts[0, :, feature_idx].detach().cpu().float().numpy()
    abs_max = float(np.abs(acts).max()) if np.abs(acts).max() > 0 else 1.0

    # Normalize to [0, 1] where 0 = no activation, 1 = strongest
    normed = np.clip(acts / abs_max, 0.0, 1.0)

    # Build highlighted HTML spans
    spans = []
    for tok, strength in zip(tokens, normed):
        # Interpolate white (255,255,255) → dark green (0,100,0)
        r = int(255 * (1 - strength))
        g = int(255 - 155 * strength)  # 255 → 100
        b = int(255 * (1 - strength))
        text_color = "#000" if strength < 0.6 else "#fff"
        tok_display = tok.replace("<", "&lt;").replace(">", "&gt;")
        spans.append(
            f'<span style="background:rgb({r},{g},{b});color:{text_color};'
            f'padding:2px 3px;margin:1px;border-radius:3px;display:inline-block;'
            f'font-family:monospace;font-size:13px;" '
            f'title="activation={acts[len(spans)]:.4f}">'
            f"{tok_display}</span>"
        )

    label = labels.get(feature_idx) or "N/A"
    html = (
        f'<div style="margin-top:12px;">'
        f'<b>Feature {feature_idx}</b> — <i>{label}</i>'
        f'<br><span style="font-size:11px;color:#666;">'
        f"Dark green = strong activation, white = weak/no activation</span>"
        f'<div style="margin-top:6px;line-height:2;">{"".join(spans)}</div>'
        f"</div>"
    )
    display(HTML(html))


# --- Example: inspect the top-ranked feature of the first token ---
example_feature = int(per_token_idxs[0, 0])
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
inspect_feature(example_feature)